In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RadarComparison"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
# Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_PRECIP_Class, RadarData_PRECIP_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#LOADING RADAR CLASS
import xesmf as xe

folderDirectory = os.path.join(
    DirectoryManager.dataDirectory,
    "Observation_Data/PRECIP/Radar",
    ModelData_NSSL.case
)

RadarData_PRECIP = RadarData_PRECIP_Class(ModelData_NSSL, folderDirectory)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Plotting import ContourPlotting_Class

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
# # import pyart
# # # https://arm-doe.github.io/pyart/API/generated/pyart.retrieve.create_cfad.html
# pyart.retrieve.create_cfad??

# # ==>

# Signature:
# pyart.retrieve.create_cfad(
#     radar,
#     field_bins,
#     altitude_bins,
#     field='reflectivity',
#     field_mask=None,
#     min_frac_thres=0.1,
# )
# Source:   
# def create_cfad(
#     radar,
#     field_bins,
#     altitude_bins,
#     field="reflectivity",
#     field_mask=None,
#     min_frac_thres=0.1,
# ):
#     """
#     This function returns a Contoured Frequency by Altitude Diagram (CFAD; Yuter et al. 1995), a 2-dimensional
#     histogram that is normalized by the number of points at each altitude. Altitude bins are masked where the counts
#     are less than a minimum fraction of the largest number of counts for any altitude row.

#     Author: Laura Tomkins (lauramtomkins@gmail.com)

#     Parameters
#     ----------
#     radar : Radar
#         Radar object used. Can be Radar or Grid object.
#     field_bins : list
#         List of bin edges for field values to use for CFAD creation.
#     altitude_bins : list
#         List of bin edges for height values to use for CFAD creation.
#     field : str
#         Field name to use to look up reflectivity data. In the
#         radar object. Default field name is 'reflectivity'.
#     field_mask : array
#         An array the same size as the field array used to mask values.
#     min_frac_thres : float, optional
#         Fraction of values to remove in CFAD normalization (default 0.1). If an altitude row has a total count that
#         is less than min_frac_thres of the largest number of total counts for any altitude row, the bins in that
#         altitude row are masked.

#     Returns
#     -------
#     freq_norm : array
#         Array of normalized frequency.
#     height_edges : array
#         Array of bin edges for height data.
#     field_edges : array of x coordinates
#         Array of bin edges for field data.

#     References
#     ----------
#     Yuter, S. E., and R. A. Houze, 1995: Three-Dimensional Kinematic and
#     Microphysical Evolution of Florida Cumulonimbus. Part II: Frequency Distributions
#     of Vertical Velocity, Reflectivity, and Differential Reflectivity. Mon. Wea. Rev.
#     123, 1941-1963. https://doi.org/10.1175/1520-0493(1995)123%3C1941:TDKAME%3E2.0.CO;2


#     """

#     # get field data
#     field_data = radar.fields[field]["data"][:]

#     # get altitude data
#     # first try to get altitude data from a radar object
#     try:
#         altitude_data = radar.gate_z["data"]
#     # if it fails, try to get altitude data from a grid object
#     except:
#         try:
#             altitude_data = radar.point_z["data"]
#         except:
#             print("No altitude data found")
#             raise

#     # option to mask data if a mask is given
#     if field_mask is not None:
#         field_data = np.ma.masked_where(field_mask, field_data)
#         altitude_data = np.ma.masked_where(field_data.mask, altitude_data)
#     else:
#         if isinstance(field_data, np.ma.MaskedArray):
#             mask = field_data.mask
#             altitude_data = np.ma.masked_where(mask, altitude_data)

#     # get raw bin counts
#     freq, height_edges, field_edges = np.histogram2d(
#         altitude_data.compressed(),
#         field_data.compressed(),
#         bins=[altitude_bins, field_bins],
#     )

#     # sum counts over y axis (height)
#     freq_sum = np.sum(freq, axis=1)
#     # get threshold for normalizing
#     point_thres = min_frac_thres * np.max(freq_sum)
#     # repeat to create array same size as freq
#     freq_sum_rep = np.repeat(freq_sum[..., np.newaxis], freq.shape[1], axis=1)
#     # normalize
#     freq_norm = freq / freq_sum_rep
#     # mask data where there is not enough points
#     freq_norm = np.ma.masked_where(freq_sum_rep < point_thres, freq_norm)

#     return freq_norm, height_edges, field_edges
# File:      ~/.local/lib/python3.10/site-packages/pyart/retrieve/cfad.py
# Type:      function

In [ ]:
def create_cfad_gridded(
    radar_gridded,
    altitude_levels,
    field_bins,
    altitude_bins,
    field_mask=None,
    min_frac_thres=0.1,
    normalize=True
):
    """
    Gridded-data version of PyART's create_cfad().
    """

    # -----------------------------
    # PyART line: field_data = radar.fields[field]["data"][:]
    # Replace with gridded data
    # -----------------------------
    field_data = np.asarray(radar_gridded)

    # -----------------------------
    # PyART line: altitude_data = radar.gate_z[...] or radar.point_z
    # Replace with broadcasted altitude array
    # -----------------------------
    nz, ny, nx = field_data.shape
    altitude_data = np.repeat(
        altitude_levels[:, None, None], ny, axis=1
    )
    altitude_data = np.repeat(altitude_data, nx, axis=2)

    # -----------------------------
    # Apply mask (PyART logic preserved)
    # -----------------------------
    if field_mask is not None:
        field_data = np.ma.masked_where(field_mask, field_data)
        altitude_data = np.ma.masked_where(field_data.mask, altitude_data)
    else:
        # If input field already masked
        if isinstance(field_data, np.ma.MaskedArray):
            mask = field_data.mask
            altitude_data = np.ma.masked_where(mask, altitude_data)
        else:
            # Mask invalid numbers (NaNs)
            field_data = np.ma.masked_invalid(field_data)
            altitude_data = np.ma.masked_where(field_data.mask, altitude_data)

    # -----------------------------
    # PyART: histogram2d using compressed() arrays
    # -----------------------------
    freq, height_edges, field_edges = np.histogram2d(
        altitude_data.compressed(),
        field_data.compressed(),
        bins=[altitude_bins, field_bins],
    )

    # =====================================================
    # RETURN RAW FREQUENCY IF normalize=False
    # =====================================================
    if not normalize:
        return freq, height_edges, field_edges

    # =====================================================
    # NORMALIZATION (PyART)
    # =====================================================
    freq_sum = np.sum(freq, axis=1)
    point_thres = min_frac_thres * np.max(freq_sum)

    freq_sum_rep = np.repeat(freq_sum[:, None], freq.shape[1], axis=1)

    freq_norm = freq / freq_sum_rep
    freq_norm = np.ma.masked_where(freq_sum_rep < point_thres, freq_norm)

    return freq_norm, height_edges, field_edges

In [ ]:
def MakeCFADCalculation(radar3d, zlevels, levels_per_bin=4, normalize=True):
    # ------------------------------
    # 1. DEFINE BINS
    # ------------------------------
    field_bins = np.arange(0, 51, 1/levels_per_bin)

    # altitude_bins = np.arange(0, radar3d.nVertLevels.size + 1)
    # zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
    zlevels_center = zlevels #both stored in center for PRECIP
    altitude_bins = zlevels

    # altitude_levels = radar3d.nVertLevels.values
    altitude_levels = zlevels_center

    # ------------------------------
    # 2. COMPUTE CFADs FOR BOTH SCHEMES
    # ------------------------------
    cfad, z_edges, dbz_edges = create_cfad_gridded(
        radar_gridded  = radar3d.values,
        altitude_levels = altitude_levels,
        field_bins      = field_bins,
        altitude_bins   = altitude_bins,
        min_frac_thres  = 0.1,
        normalize=normalize
    )

    # ------------------------------
    # 3. BIN CENTERS FOR PLOTTING
    # ------------------------------
    z_centers   = 0.5 * (z_edges[:-1] + z_edges[1:])
    dbz_centers = 0.5 * (dbz_edges[:-1] + dbz_edges[1:])

    return cfad, z_centers, dbz_centers

def NormalizeCFAD(freq, min_frac_thres=0.1):
    # =====================================================
    # NORMALIZATION (PyART)
    # =====================================================
    freq_sum = np.sum(freq, axis=1)
    point_thres = min_frac_thres * np.max(freq_sum)

    freq_sum_rep = np.repeat(freq_sum[:, None], freq.shape[1], axis=1)

    freq_norm = freq / freq_sum_rep
    freq_norm = np.ma.masked_where(freq_sum_rep < point_thres, freq_norm)

    return freq_norm

In [ ]:
####################################
#DATA LOADING FUNCTIONS

In [ ]:
def GetFileNamePath_PRECIP(ModelData):
    
    # Build file name
    fileName = (
        f"PRECIP_CFAD_{ModelData.region}_"
        f"{ModelData.case}_spinup{ModelData.spinup_hours}hrs.pkl"
    )
    
    # Build directory for radar timeseries
    outputDir = os.path.join(
        DirectoryManager.GetOutputDirectory(codeType, dataType),
        "ComparingCFADs"
    )

    os.makedirs(outputDir, exist_ok=True)
    
    # Full path to the .pkl file
    fileNamePath = os.path.join(outputDir, fileName)
    return fileNamePath

In [ ]:
#Converting timeStrings
def ConvertTimeStringtoDateTime(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    """
    return datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
def ConvertTimeStringtoTimeTitle(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    Formatted for use as a plot title.
    """
    # Parse the custom format to a datetime object
    dt = datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
    # Format it to 'YYYY-MM-DD HH:MM:SS'
    return dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
def RunCalculations(
    fileNamePath,
    ModelData_NSSL,
    DirectoryManager,
    min_frac_thres=0.1):
    """
    Load CFADs from pickle file, or compute, save, and return them.
    """

    # ----------------------------------------------------------
    # 1. LOAD EXISTING FILE IF PRESENT
    # ----------------------------------------------------------
    if os.path.exists(fileNamePath):
        print(f"Loading precomputed CFADs from: {fileNamePath}")
        with open(fileNamePath, "rb") as f:
            return pickle.load(f)

    print(f"CFAD file {fileNamePath} not found. Computing CFADs from scratch...")

    # ----------------------------------------------------------
    # 2. SETUP
    # ----------------------------------------------------------
    total_CFAD_PRECIP  = None

    # Load radar mask once
    mask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData_NSSL)

    # Determine which times to process
    time_range=None
    times = time_range if time_range is not None else range(ModelData_NSSL.Ntime)

    # Loading zlevels
    zlevels = RadarData_PRECIP.z_heights

    # ----------------------------------------------------------
    # 3. LOOP OVER TIME
    # ----------------------------------------------------------
    for t in tqdm(times, desc="Computing CFADs"):
        # ------------------------------
        # LOAD DATA
        # ------------------------------

        radarData_PRECIP, _ = RadarData_PRECIP.GetData_AllZLevels(DirectoryManager, ModelData_NSSL,t) #no interpolation is done for PRECIP since interpolation routine is too slow
        radarData_PRECIP =  RadarData_PRECIP.InterpolateRadarData3D(radarData_PRECIP, ModelData_NSSL, DirectoryManager)
        radarData_PRECIP = radarData_PRECIP.where(radarData_PRECIP>0)

        # radarData_PRECIP  = radarData_PRECIP.where(mask) #not needed

        # ------------------------------
        # RAW HISTOGRAM (not normalized)
        # ------------------------------
        raw_PRECIP,  z_centers, dbz_centers = MakeCFADCalculation(radarData_PRECIP, zlevels, normalize=False)

        # ------------------------------
        # ACCUMULATE
        # ------------------------------
        if total_CFAD_PRECIP is None:
            total_CFAD_PRECIP  = raw_PRECIP.copy()
        else:
            total_CFAD_PRECIP  += raw_PRECIP

    # ----------------------------------------------------------
    # 4. NORMALIZE
    # ----------------------------------------------------------
    CFAD_PRECIP  = NormalizeCFAD(total_CFAD_PRECIP,  min_frac_thres=min_frac_thres)

    # ----------------------------------------------------------
    # 5. NORMALIZE RAW OUTPUT
    # ----------------------------------------------------------
    total_CFAD_PRECIP = np.ma.masked_where(total_CFAD_PRECIP == 0, total_CFAD_PRECIP)

    # ----------------------------------------------------------
    # COMBINING INTO DICTIONARY
    # ----------------------------------------------------------
    results = {
        # --- RAW (unnormalized) CFADs ---
        "CFAD_PRECIP_raw": total_CFAD_PRECIP,
        
        # --- NORMALIZED CFADs ---
        "CFAD_PRECIP": CFAD_PRECIP,

        # --- AXES ---
        "z_centers": z_centers,
        "dbz_centers": dbz_centers}

    # ----------------------------------------------------------
    # SAVE TO PKL
    # ----------------------------------------------------------
    with open(fileNamePath, "wb") as f:
        pickle.dump(results, f)

    print(f"Saved CFADs to: {fileNamePath}")

    return results

In [ ]:
fileNamePath = GetFileNamePath_PRECIP(ModelData=ModelData_NSSL)
CFAD_results = RunCalculations(
    fileNamePath=fileNamePath,
    ModelData_NSSL=ModelData_NSSL,
    DirectoryManager=DirectoryManager)

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def GetCFADColormap():

    # ---------------------------------------
    # 1. Define the LOG bin boundaries
    #    (you can adjust these if needed)
    # ---------------------------------------
    log_bins = np.array([1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2])

    # ---------------------------------------
    # 2. Define colors for each log interval
    #    (10 colors per log bin)
    # ---------------------------------------
    n_per_group = 10   # number of colors per log interval

    # These are the target group colors:
    grey     = np.array([0.60, 0.60, 0.60])
    green    = np.array([0.20, 0.70, 0.20])
    blue     = np.array([0.20, 0.40, 0.90])
    yellow   = np.array([0.95, 0.85, 0.20])
    orange   = np.array([1.00, 0.50, 0.05])
    red      = np.array([0.90, 0.10, 0.10])
    darkred  = np.array([0.60, 0.00, 0.00])

    # Ordered from low → high intensity
    group_colors = [grey, green, blue, yellow, orange, red, darkred]

    # ---------------------------------------
    # 3. Interpolate each group into 10 steps
    # ---------------------------------------
    colors_per_bin = []
    for i in range(len(group_colors)-1):
        start = group_colors[i]
        end   = group_colors[i+1]

        # generate gradient between two colors
        grad = np.linspace(start, end, n_per_group)
        colors_per_bin.append([tuple(c) for c in grad])

    # Flatten color list
    all_colors = [c for group in colors_per_bin for c in group]

    # ---------------------------------------
    # 4. Expand the log bins into sub-bins
    #    (10 subdivisions per decade)
    # ---------------------------------------
    expanded_bins = []
    for low, high in zip(log_bins[:-1], log_bins[1:]):
        sub = np.logspace(np.log10(low), np.log10(high), n_per_group + 1)
        expanded_bins.extend(sub[:-1])
    expanded_bins.append(log_bins[-1])
    expanded_bins = np.array(expanded_bins)

    # ---------------------------------------
    # 5. Build the colormap + norm
    # ---------------------------------------
    cmap = mcolors.ListedColormap(all_colors)
    norm = mcolors.BoundaryNorm(expanded_bins, len(all_colors))

    return cmap, norm

def GetCFADDiffColormap():

    # ---------------------------------------
    # 1. Define symmetric difference bins
    # ---------------------------------------
    diff_edges = np.array([-100, -10, -1, -0.1, 0, 0.1, 1, 10, 100])

    # ---------------------------------------
    # 2. Same colors as main CFAD colormap
    # ---------------------------------------
    n_per_group = 10

    grey     = np.array([0.60, 0.60, 0.60])
    green    = np.array([0.20, 0.70, 0.20])
    blue     = np.array([0.20, 0.40, 0.90])
    yellow   = np.array([0.95, 0.85, 0.20])
    orange   = np.array([1.00, 0.50, 0.05])
    red      = np.array([0.90, 0.10, 0.10])
    darkred  = np.array([0.60, 0.00, 0.00])

    group_colors = [grey, green, blue, yellow, orange, red, darkred]

    # ---------------------------------------
    # 3. Build the same 60-color list
    # ---------------------------------------
    colors_per_bin = []
    for i in range(len(group_colors)-1):
        grad = np.linspace(group_colors[i], group_colors[i+1], n_per_group)
        colors_per_bin.append([tuple(c) for c in grad])

    all_colors = [c for group in colors_per_bin for c in group]  # 60 colors

    # ---------------------------------------
    # 4. Build expanded bins matching 60 colors
    #    → evenly interpolate the 8 intervals to produce 60 bins
    # ---------------------------------------

    # We need exactly 60 boundaries for BoundaryNorm
    Ncolors = len(all_colors)
    Nedges = Ncolors + 1  # = 61

    # Interpolate across the full difference range
    expanded_bins = np.interp(
        np.linspace(0, len(diff_edges)-1, Nedges),
        np.arange(len(diff_edges)),
        diff_edges
    )

    # ---------------------------------------
    # 5. Build colormap + boundary norm
    # ---------------------------------------
    cmap = mcolors.ListedColormap(all_colors)
    norm = mcolors.BoundaryNorm(expanded_bins, Ncolors)

    return cmap, norm


In [ ]:
def PlotCFAD_PRECIP(CFAD_results, cmap='turbo', norm=None, colorbar_pad=0.01, normalize = True):
    """
    Plot a single CFAD for PRECIP reflectivity.
    """

    # ------------------------------
    # 1. LOAD CFAD RESULTS
    # ------------------------------
    if normalize == True:
        cfad = CFAD_results["CFAD_PRECIP"] * 1e2   # convert to %
    else:
        cfad = CFAD_results["CFAD_PRECIP_raw"]
        norm=None
    z_centers   = CFAD_results["z_centers"]
    dbz_centers = CFAD_results["dbz_centers"]

    #Getting first All Nan Levels
    lowestNanLevel  = FirstNanLevel(cfad)

    # ------------------------------
    # 2. FIGURE
    # ------------------------------
    fig, ax = plt.subplots(figsize=(7, 8))

    # ------------------------------
    # 3. CONTOUR LEVELS
    # ------------------------------
    if norm is not None:
        levels = norm.boundaries
    else:
        levels = 100

    # ------------------------------
    # 4. PLOT PRECIP CFAD
    # ------------------------------
    cf = ax.contourf(
        dbz_centers, z_centers, cfad,
        cmap=cmap, norm=norm,
        levels=levels, extend='both'
    )

    ax.set_title("PRECIP Reflectivity CFAD")
    ax.set_xlabel("Reflectivity (dBZ)")
    ax.set_ylabel("Altitude (km)")

    # ------------------------------
    # 5. TRIM EMPTY ALTITUDE ROWS
    # ------------------------------
    z_top=15
    ax.set_ylim(0, z_top)

    # ------------------------------
    # 6. COLORBAR
    # ------------------------------
    if norm is None:
        cbar = fig.colorbar(cf, ax=ax, pad=colorbar_pad)
    else:
        cbar = fig.colorbar(
            cf, ax=ax, pad=colorbar_pad,
            boundaries=norm.boundaries, extend='both'
        )
        # optional log-style labels if using log norm
        tick_vals = [1e-3, 1e-2, 1e-1, 1, 10, 100]
        cbar.set_ticks(tick_vals)
        cbar.set_ticklabels(['1e−3','1e−2','1e−1','1','10','100'])

    if normalize == True:
        cbar.set_label("Normalized Frequency (%)")
    else:
        cbar.set_label("Count")

    return fig


def FirstNanLevel(array):
    levelAllNan = np.all(np.isnan(array), axis=1)
    idx = np.where(levelAllNan)[0]
    return idx[0] if idx.size > 0 else None

In [ ]:
####################################
#PLOTTING

In [ ]:
custom_cmap, custom_norm = GetCFADColormap()
custom_cmap_diff, custom_norm_diff = GetCFADDiffColormap()

fig = PlotCFAD_PRECIP(
    CFAD_results,
    cmap=custom_cmap,
    norm=custom_norm,
    colorbar_pad=0.01,
)

#This plot will be added to ComparingCFADs.ipynb

In [ ]:
custom_cmap, custom_norm = GetCFADColormap()
custom_cmap_diff, custom_norm_diff = GetCFADDiffColormap()

fig = PlotCFAD_PRECIP(
    CFAD_results,
    cmap=custom_cmap,
    norm=None,
    colorbar_pad=0.01,
    normalize=False
)

#This plot will be added to ComparingCFADs.ipynb